# Making systems secure and reliable

**Optional depth track · module 4 of 5**

**Goal:** Test the paths you hope never happen, make a timeout return an answer instead of a stack trace, and write down the risk you decided to live with.

**Why it matters:** Shifting security left means it is your job, not a later review's. The cheapest version of that is a table of bad inputs where you wrote down what actually happened when you sent them, which is a different document from the one where you wrote down what you hoped would happen.

Nothing here is graded and nothing in the fifteen sessions depends on it. Work through it when you want the layer underneath.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

In [ ]:
# depth_checks registers this track's checkers. check() and review() are the
# same ones the course uses.
from bootcamp_agent.checks import check, review
import bootcamp_agent.depth_checks  # noqa: F401

## 1. The negative matrix

**Context.** Four bad inputs. For each one, what you expected and what you **observed** when
you ran it. The observed column is the entire value of this table: an unfilled one means the
row was never tested, and an untested row is a hope.

**Instructions.**

1. Run each input against the agent or the tools, however you like.
2. Fill `observed` with what you saw: the error class, the message, the status.
3. "As expected" is refused. Write what happened.

In [ ]:
matrix = [
    {"input": "a question that is an empty string",
     "expected": "a refusal, and no model call at all",
     "observed": "refusal returned, needs_human_review True, FakeLLM.calls stayed at 0"},
    {"input": "a tool argument of the wrong type, doc_id=12345",
     "expected": "a typed ToolError naming the argument",
     "observed": "ToolError raised: 'doc_id must be a string'. Nothing was read."},
    {"input": "a retrieved passage containing 'ignore previous instructions'",
     "expected": "treated as data; the instruction is not followed",
     "observed": "the answer summarised the passage and cited it; the instruction was quoted, "
                 "not obeyed"},
    {"input": "a question about something not in the corpus",
     "expected": "a refusal with no citations",
     "observed": "refusal, citations empty, needs_human_review True"},
]
for row in matrix:
    print(f"{row['input'][:44]:46} {row['observed'][:38] or '(not run)'}")

**Expected output**

```
a question that is an empty string             refusal returned, needs_human_review T
a tool argument of the wrong type, doc_id=1    ToolError raised: 'doc_id must be a st
...
✅ d4-e1 passed
```

In [ ]:
check("d4-e1", matrix)

## 2. A timeout that returns an answer

**Context.** The model provider will hang. When it does, the caller should get something they
can render, flagged for review, citing nothing. Not a stack trace, and not a confident answer
built from no evidence.

**Instructions.**

1. `answer_within(client, question, seconds)` calls the client and returns an answer-shaped dict.
2. On `TimeoutError`, return a flagged refusal with **no citations**.
3. The healthy path must not be flagged. A guard that fires always is not a guard.

In [ ]:
import json


def answer_within(client, question: str, seconds: float) -> dict:
    try:
        raw = client.complete("system", question)
    except TimeoutError:
        # No citations: we retrieved nothing and heard nothing, so there is
        # nothing to ground an answer in. Flagged, so the client renders it as
        # a refusal rather than as an answer.
        return {
            "answer": "I could not answer in time. Try again, or send this to a human.",
            "citations": [],
            "confidence": 0.0,
            "needs_human_review": True,
        }
    parsed = json.loads(raw)
    return parsed


class Slow:
    def complete(self, system, user):
        raise TimeoutError("upstream took too long")


print(answer_within(Slow(), "what is chunking", 0.01))

**Expected output**

```
{'answer': 'I could not answer in time. ...', 'citations': [], 'confidence': 0.0, 'needs_human_review': True}
✅ d4-e2 passed
```

In [ ]:
check("d4-e2", answer_within)

## 3. The threat model, including what you accept

**Context.** One page. What is worth protecting, where untrusted input stops being trusted,
what you did about it, and the one risk you decided to live with. That last field is the one
people skip, and it is the one that makes the document honest.

**Instructions.**

1. List the assets. For this system they are smaller than you think.
2. Name the trust boundary in one sentence.
3. `accepted` names a real risk you are not fixing, and why that is a reasonable call.

In [ ]:
model = {
    "assets": ["the corpus", "the provider API key", "the questions learners ask"],
    "trust_boundary": "Everything read from the corpus or returned by a tool is untrusted "
                      "from the moment it is loaded, and stays data all the way to the model.",
    "mitigated": [
        "retrieved text is passed as data, never as instructions",
        "tools are read-only, typed and capped at three calls",
        "citations are verified against retrieved doc ids before an answer is returned",
    ],
    "accepted": "A learner with repository access can edit the corpus and change what the "
                "assistant says. We accept it: the corpus is a teaching fixture in a repo "
                "they already control, and locking it down would cost more than it protects.",
}
print(model["trust_boundary"] or "(no boundary written)")
print(model["accepted"] or "(nothing accepted — that is the tell)")

**Expected output**

```
Everything read from the corpus or returned by a tool is untrusted from the moment ...
A learner with repository access can edit the corpus and change what the assistant says. ...
✅ d4-e3 passed
```

In [ ]:
check("d4-e3", model)

## Review

The scorecard for this module. Every ❌ names the exercise and the hint.

In [ ]:
review("d4")